# 推計気象分布と解析雨量の地点切り出しデータ作成
- メッシュ農業気象データとの比較検証用データセット作成

## 前準備

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import GPVdecoder as decoder

In [2]:
### 変数設定 ###
#--圃場リスト
farm_topriver = "farm/topriver_farm.csv"
farm_jpagri = "farm/jpagri_farm.csv"
farm_kayano = "farm/kayano_farm.csv"

#--ディレクトリ
wx_anal_base = "wx_anal/{:04d}{:02d}"
ra_anal_base = "ra_anal/{:04d}{:02d}"
csv_dir = "org_csv"
out_dir = "wm_data"

#--期間
# 推計気象分布のファイル名の時刻はUTCな点に注意
# start_date = "2022/12/31 16:00:00"
# end_date = "2025/10/31 15:00:00"
start_date = "2022/12/31"
end_date = "2025/11/30"
# start_date = "2025/11/01"
# end_date = "2025/11/30"

#--データ
# wx_anal_name = "OBS"

In [3]:
### 前処理 ###
#--店舗情報（店舗コード、緯度経度）
farm1 = pd.read_csv(farm_topriver)
farm2 = pd.read_csv(farm_jpagri)
farm3 = pd.read_csv(farm_kayano)
use_cols = ['field_id', 'latitude', 'longitude']
farm = pd.concat([farm1[use_cols], farm2[use_cols], farm3[use_cols]], ignore_index=True)
farm.rename(columns={'field_id':'code'}, inplace=True)

#--ベースタイムのリスト
# date_list = pd.date_range(start_date, end_date, freq="1h")
date_list = pd.date_range(start_date, end_date, freq="1d")

## 推計気象分布データ地点切り出し
- GPVデータ（推計気象分布）から各店舗の地点データを切り出す
- CSVに出力する

In [4]:
### 地点切り出しのループ処理 ###
data = {}
for date in date_list:
    # print(date)

    gpv_dir = wx_anal_base.format(date.year, date.month)
    # date_str = date.strftime(format='%Y%m%d%H')
    # gpv_files = glob.glob(f"{gpv_dir}/Z__C_RJTD_{date_str}0000*")
    date_str = date.strftime(format='%Y%m%d')
    gpv_files = glob.glob(f"{gpv_dir}/Z__C_RJTD_{date_str}*")

    if len(gpv_files) == 0:
        print(f"skip {date}")
        continue

    #--地点切り出し
    gpv_obs = decoder.GPV(basetime=date, model="OBS", gpvfiles=gpv_files, points=farm, grib2table="tbl/obs_gpv.grib2table")
    gpv_obs.get_gpv()

    for sid in gpv_obs.data.keys():
        if sid in data.keys():
            data[sid] = pd.concat([data[sid], gpv_obs.data[sid]], axis=0)
        else:
            data[sid] = gpv_obs.data[sid]

In [5]:
#--日単位集計前の生データを出力
os.makedirs(csv_dir, exist_ok=True)
for sid in data.keys():
    df = data[sid].copy()
    #--WXCATとQIは整数値で出力する（欠損を含みうる）
    df["WXCAT_SFC"] = df["WXCAT_SFC"].astype("Int64")
    df["QI_SFC"] = df["QI_SFC"].astype("Int64")
    #--書き出し
    file = os.path.join(csv_dir, f"wx_anal_{sid}.csv")
    df.to_csv(file, index=False)

## 解析雨量データ地点切り出し

In [4]:
### 地点切り出しのループ処理 ###
data = {}
for date in date_list:
    # print(date)

    gpv_dir = ra_anal_base.format(date.year, date.month)
    # date_str = date.strftime(format='%Y%m%d%H')
    # gpv_files = glob.glob(f"{gpv_dir}/Z__C_RJTD_{date_str}0000*")
    date_str = date.strftime(format='%Y%m%d')
    gpv_files = glob.glob(f"{gpv_dir}/Z__C_RJTD_{date_str}*0000_*")
    # print(gpv_files)

    if len(gpv_files) == 0:
        print(f"skip {date}")
        continue

    #--地点切り出し
    gpv_obs = decoder.GPV(basetime=date, model="OBS", gpvfiles=gpv_files, points=farm, grib2table="tbl/obs_gpv.grib2table")
    gpv_obs.get_gpv()

    for sid in gpv_obs.data.keys():
        if sid in data.keys():
            data[sid] = pd.concat([data[sid], gpv_obs.data[sid]], axis=0)
        else:
            data[sid] = gpv_obs.data[sid]

In [5]:
#--日単位集計前の生データを出力
os.makedirs(csv_dir, exist_ok=True)
for sid in data.keys():
    df = data[sid].copy()
    #--書き出し
    file = os.path.join(csv_dir, f"ra_anal_{sid}.csv")
    df.to_csv(file, index=False)

## Dailyデータへ加工処理
- メッシュ農業気象データ利用時のフォーマットに合わせて日単位のデータを作る
    - 平均気温
    - 最高気温
    - 最低気温
    - 日降水量
    - 日照時間
    - 雨が降った時間数

In [4]:
### 日単位の気象データ作成 ###
#--出力先ディレクトリ
os.makedirs(out_dir, exist_ok=True)

#--地点ごとの処理
for code in farm['code'].values:
    df_wx = pd.read_csv(os.path.join(csv_dir, f"wx_anal_{code}.csv"), parse_dates=['VALIDTIME'])
    df_ra = pd.read_csv(os.path.join(csv_dir, f"ra_anal_{code}.csv"), parse_dates=['VALIDTIME'])
    df = pd.merge(df_wx.drop('FT',axis=1), df_ra.drop('FT',axis=1), on='VALIDTIME', how='outer')

    #--FTを削除し、気象要素のカラム名から_SFCを削除
    # df.drop(columns=['FT'], inplace=True)
    df.columns = [col.replace('_SFC','') for col in df.columns]

    #--日照時間のQC
    df['SUNSD'] = df['SUNSD'].mask(df['QI'] == 128, np.nan)

    #--時刻を日本時間に変換するが、1時〜24時のデータを1日として扱う
    df['VALIDTIME'] = df['VALIDTIME'] + pd.Timedelta(hours=9)
    df['date'] = (df['VALIDTIME'] - pd.Timedelta(hours=1)).dt.date
    df['hour'] = (df['VALIDTIME'] - pd.Timedelta(hours=1)).dt.hour + 1

    #--降水があるかどうか（雨・みぞれ・雪）
    df['is_PRCP'] = (df['WXCAT'] >= 3).astype(int)

    #--1日ごとの統計量
    df_daily = df.groupby('date').agg(
        TMP_mea=('TMP', 'mean'),
        TMP_max=('TMP', 'max'),
        TMP_min=('TMP', 'min'),
        APCPRA=('PRCP', 'sum'),
        SSD=('SUNSD', 'sum'),
        APCP_hours=('is_PRCP', 'sum'),
    ).round(1)

    #--CSV出力
    # 最初と最後の日はデータが不完全な場合があるので除外する
    df_daily.iloc[1:-1].to_csv(os.path.join(out_dir, f"wmdata_{code}.csv"))